In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#      (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report

ROOT = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')

GENRES = ["blues", "classical", "country", "disco",
          "hiphop", "jazz", "metal", "pop", "reggae", "rock"]


In [3]:
# Q1: Mean Duration of Jazz stems
# Use soundfile.info() to read only metadata (no audio decoding) - much faster
jazz_path = os.path.join(STEMS_PATH, "jazz")

durations = []

for song in tqdm(os.listdir(jazz_path)):
    song_path = os.path.join(jazz_path, song)

    if os.path.isdir(song_path):
        for stem in os.listdir(song_path):
            file_path = os.path.join(song_path, stem)
            if file_path.endswith('.wav') and os.path.exists(file_path):
                try:
                    info = sf.info(file_path)
                    durations.append(info.duration)
                except:
                    continue

print("Answer Q1 (Mean Duration Jazz):", np.mean(durations))


100%|██████████| 100/100 [00:01<00:00, 55.08it/s]

Answer Q1 (Mean Duration Jazz): 30.032979591836728


In [4]:
# Q2: Unique Sample Rates across all genres
# soundfile.info() reads only header - extremely fast vs librosa.load()
sample_rates = set()

# edit this code to consider genre_stems, noise data and messy_mashups (in order) # [22050, 44100]

for g in GENRES:
    genre_path = os.path.join(STEMS_PATH, g)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if not os.path.isdir(song_path):
            continue
        for stem in os.listdir(song_path):
            file_path = os.path.join(song_path, stem)
            if file_path.endswith('.wav') and os.path.exists(file_path):
                try:
                    info = sf.info(file_path)
                    sample_rates.add(info.samplerate)
                except:
                    continue

print("Answer Q2 (Unique Sample Rates):", sorted(list(sample_rates)))


Answer Q2 (Unique Sample Rates): [44100]


In [5]:
# Q3: Corrupted (zero-byte) files - already fast, no change needed
corrupted_count = 0

for g in GENRES:
    genre_path = os.path.join(STEMS_PATH, g)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if not os.path.isdir(song_path):
            continue
        for stem in os.listdir(song_path):
            file_path = os.path.join(song_path, stem)
            if os.path.exists(file_path) and os.path.getsize(file_path) == 0:
                corrupted_count += 1

print("Answer Q3 (Corrupted Files):", corrupted_count)


Answer Q3 (Corrupted Files): 0


In [6]:
# Q4: Average Peak dB of Vocals
# Use soundfile.read() instead of librosa.load() - avoids resampling overhead
def get_peak_db(vocal_path):
    try:
        y, sr = sf.read(vocal_path, dtype='float32')
        if y.ndim > 1:
            y = y.mean(axis=1)  # stereo -> mono
        if len(y) == 0:
            return None
        peak = np.max(np.abs(y))
        return 20 * np.log10(peak + 1e-10)
    except:
        return None

vocal_paths = []
for g in GENRES:
    genre_path = os.path.join(STEMS_PATH, g)
    for song in os.listdir(genre_path):
        vocal_path = os.path.join(genre_path, song, "vocals.wav")
        if os.path.exists(vocal_path):
            vocal_paths.append(vocal_path)

# Parallel processing
peak_db_values = []
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(get_peak_db, p): p for p in vocal_paths}
    for f in tqdm(as_completed(futures), total=len(futures)):
        result = f.result()
        if result is not None:
            peak_db_values.append(result)

print("Answer Q4 (Avg Peak dB Vocals):", np.mean(peak_db_values))


100%|██████████| 1000/1000 [00:13<00:00, 72.10it/s]

Answer Q4 (Avg Peak dB Vocals): -12.494921


In [7]:
# Q5 & Q6: Spectral Centroid - compute all genres together (avoid repeated I/O)
# Use librosa with mono=True and fixed sr to skip resampling where possible

def get_centroid(file_path):
    try:
        y, sr = librosa.load(file_path, sr=22050, mono=True)
        if len(y) == 0:
            return None
        return float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    except:
        return None

genre_centroids = {}
blues_centroids = []

for g in GENRES:
    genre_path = os.path.join(STEMS_PATH, g)
    file_paths = []
    for song in os.listdir(genre_path):
        for fname in ['other.wav', 'others.wav']:
            fp = os.path.join(genre_path, song, fname)
            if os.path.exists(fp):
                file_paths.append(fp)
                break

    centroids = []
    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = {ex.submit(get_centroid, p): p for p in file_paths}
        for f in tqdm(as_completed(futures), total=len(futures), desc=g):
            r = f.result()
            if r is not None:
                centroids.append(r)

    if centroids:
        genre_centroids[g] = np.mean(centroids)
        if g == 'blues':
            blues_centroids = centroids

print("Answer Q5 (Mean Spectral Centroid Blues):", genre_centroids.get('blues', 'N/A'))
print("All Genre Means:", genre_centroids)
print("Answer Q6 (Highest Centroid Genre):", max(genre_centroids, key=genre_centroids.get))


rock: 100%|██████████| 100/100 [00:04<00:00, 23.53it/s]

Answer Q5 (Mean Spectral Centroid Blues): 1597.8387497380847
All Genre Means: {'blues': np.float64(1597.8387497380847), 'classical': np.float64(1357.8935316163452), 'country': np.float64(1363.486739099068), 'disco': np.float64(1798.6066545573412), 'hiphop': np.float64(2429.2962754907903), 'jazz': np.float64(1554.8343991374256), 'metal': np.float64(2418.8018854941583), 'pop': np.float64(1517.4679898262732), 'reggae': np.float64(1761.585430812623), 'rock': np.float64(1547.635900011739)}
Answer Q6 (Highest Centroid Genre): hiphop


In [8]:
# Q7: Files with silent first 0.5 seconds
# Key optimization: load only 0.5s of audio using duration= parameter

def check_silence(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None, mono=True, duration=0.5)
        if len(y) == 0:
            return False
        return np.max(np.abs(y)) < 1e-4
    except:
        return False

all_files = []
for g in GENRES:
    genre_path = os.path.join(STEMS_PATH, g)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if not os.path.isdir(song_path):
            continue
        for stem in os.listdir(song_path):
            fp = os.path.join(song_path, stem)
            if fp.endswith('.wav') and os.path.exists(fp):
                all_files.append(fp)

silence_count = 0
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(check_silence, p): p for p in all_files}
    for f in tqdm(as_completed(futures), total=len(futures)):
        if f.result():
            silence_count += 1

print("Answer Q7 (Silence Count):", silence_count)


100%|██████████| 4000/4000 [00:04<00:00, 997.36it/s] 

Answer Q7 (Silence Count): 333


In [9]:
# Feature extraction function - unchanged logic, will parallelize below
def extract_features_safe(song_path):
    possible_files = ['other.wav', 'others.wav']
    file_path = None

    for fname in possible_files:
        temp_path = os.path.join(song_path, fname)
        if os.path.exists(temp_path):
            file_path = temp_path
            break

    if file_path is None:
        return None

    try:
        y, sr = librosa.load(file_path, sr=22050, duration=10, mono=True)

        if len(y) == 0:
            return None

        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

        return [float(tempo), spec_cent, zcr, rolloff]

    except:
        return None


In [10]:
# Build dataset
data = []
for g in GENRES:
    gp = os.path.join(STEMS_PATH, g)
    songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
    for s in songs[:50]:
        data.append({'path': os.path.join(gp, s), 'genre': g})

df = pd.DataFrame(data)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['genre'],
    random_state=42
)


In [11]:
# Parallel feature extraction using ThreadPoolExecutor
def extract_with_genre(row):
    path, genre = row['path'], row['genre']
    features = extract_features_safe(path)
    return (features, genre)

def parallel_extract(df_subset):
    rows = [row for _, row in df_subset.iterrows()]
    X, y = [], []
    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = {ex.submit(extract_with_genre, row): row for row in rows}
        for f in tqdm(as_completed(futures), total=len(futures)):
            features, genre = f.result()
            if features is not None:
                X.append(features)
                y.append(genre)
    return np.array(X), y

print("Extracting train features...")
X_train, y_train = parallel_extract(train_df)

print("Extracting validation features...")
X_val, y_val = parallel_extract(val_df)

print("Train samples:", len(X_train))
print("Validation samples:", len(X_val))


Extracting train features...


  0%|          | 0/400 [00:00<?, ?it/s]/tmp/ipykernel_56/3117547627.py:26: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return [float(tempo), spec_cent, zcr, rolloff]
100%|██████████| 400/400 [00:26<00:00, 15.31it/s]


Extracting validation features...


100%|██████████| 100/100 [00:04<00:00, 20.35it/s]

Train samples: 400
Validation samples: 100


In [12]:
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=5, random_state=42)

In [13]:
y_pred = clf.predict(X_val)

macro_f1 = f1_score(y_val, y_pred, average='macro')
print("Answer Q8 (Validation Macro F1):", macro_f1)

Answer Q8 (Validation Macro F1): 0.15230042016806722


In [14]:
cr = classification_report(y_val, y_pred, output_dict=True)

print("Answer Q9 (Precision hiphop):", cr['hiphop']['precision'])

Answer Q9 (Precision hiphop): 0.25


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
print("Answer Q10 (Recall pop):", cr['pop']['recall'])

Answer Q10 (Recall pop): 0.2


In [16]:
accuracy = np.mean(y_pred == y_val)
print("Answer Q11 (Accuracy):", accuracy)

Answer Q11 (Accuracy): 0.19


In [17]:
cm = confusion_matrix(y_val, y_pred, labels=GENRES)

tp_dict = {}

for i, genre in enumerate(GENRES):
    TP = cm[i, i]
    tp_dict[genre] = TP

print("Answer Q12 (Highest TP Genre):", max(tp_dict, key=tp_dict.get))

Answer Q12 (Highest TP Genre): metal


In [18]:
fn_dict = {}

for i, genre in enumerate(GENRES):
    TP = cm[i, i]
    FN = np.sum(cm[i, :]) - TP
    fn_dict[genre] = FN

print("Answer Q13 (Lowest FN Genre):", min(fn_dict, key=fn_dict.get))

Answer Q13 (Lowest FN Genre): metal
